# Round 5 — เทรนบน Google Colab (GPU)

เทรนโมเดล Round 5 (augmentation แบบสุ่มระหว่างเทรน) บน Colab · ใช้ข้อมูลของอาจารย์เท่านั้น (ไม่ใช้ข้อมูลภายนอก)

**ก่อนรัน**
1. เมนู *Runtime → Change runtime type* เลือก **GPU (T4)**
2. อัปโหลด `ThaiCharacter Dataset.zip` ไว้ใน Google Drive แล้วแก้ path ในเซลล์ "ตั้งค่า" ให้ตรง
3. รันเซลล์ตามลำดับ (ผลเทรนถูกเขียนลง Drive โดยตรง ไม่หายเมื่อรันไทม์หลุด)

**หมายเหตุ:** โน้ตบุ๊กนี้เขียนจากโค้ดที่อ่านมา *ยังไม่ได้ทดสอบบน Colab จริง* ถ้าเซลล์ไหนพัง ให้ส่งข้อความ error มา
ไม่มีระบบเทรนต่อจาก checkpoint (`TrainingCNN.py` ของ Round 5 ไม่มี resume) ถ้ารันไทม์หลุดกลางทางต้องเริ่มรอบนั้นใหม่

## 1) ตรวจ GPU

In [1]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี GPU — ตรวจ Runtime type")

torch 2.11.0+cu128 | cuda: True | Tesla T4


## 2) เชื่อม Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3) ตั้งค่า (แก้ path ให้ตรงกับ Drive ของคุณ)

In [3]:
import os
DRIVE_ZIP  = "/content/drive/MyDrive/Deep Learning/ThaiCharacter Dataset.zip"   # ไฟล์ข้อมูลของอาจารย์
DRIVE_RUNS = "/content/drive/MyDrive/Deep Learning/round5_runs"                # ผลเทรนจะถูกเขียนที่นี่
REPO_URL   = "https://github.com/Synthesizzz/thai-character-recognition.git"

assert os.path.isfile(DRIVE_ZIP), f"ไม่พบไฟล์ข้อมูล: {DRIVE_ZIP}"
print("พบไฟล์ข้อมูลแล้ว")

พบไฟล์ข้อมูลแล้ว


## 4) ดึงโค้ดจาก GitHub (เฉพาะโฟลเดอร์ `Round 5`)

ใช้ sparse checkout เพื่อไม่ต้องโหลดไฟล์น้ำหนัก `model.pt` ของรอบอื่น (รวมหลายร้อย MB)
ถ้า repo เป็น private จะ clone ไม่ได้ ต้องใช้ token

In [4]:
%cd /content
!rm -rf repo
!git clone --depth 1 --filter=blob:none --sparse {REPO_URL} repo
%cd /content/repo
!git sparse-checkout set "Round 5"
!ls "Round 5"

/content
Cloning into 'repo'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 20 (delta 0), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 10.13 KiB | 10.13 MiB/s, done.
/content/repo
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 7 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), 129.34 KiB | 3.40 MiB/s, done.
evaluate_synthetic.py  Net.py  README.md  runs	TestingCNN.py  TrainingCNN.py


## 5) แตกไฟล์ข้อมูล

zip มีโครงสร้าง `round2/<รหัสคลาส>/*.jpg` (และโฟลเดอร์ขยะ `__MACOSX` ที่ข้ามไป) โค้ดต้องการ
`ThaiCharacter Dataset/round2/...` อยู่ข้างโฟลเดอร์ `Round 5`

In [5]:
!mkdir -p "/content/repo/ThaiCharacter Dataset"
!unzip -q "{DRIVE_ZIP}" -d "/content/repo/ThaiCharacter Dataset" -x "__MACOSX/*"

d = "/content/repo/ThaiCharacter Dataset/round2"
print("จำนวนคลาส:", len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))]), "(ต้องได้ 72)")

จำนวนคลาส: 72 (ต้องได้ 72)


## 6) ให้ผลเทรนเขียนลง Drive โดยตรง

ทำให้ `Round 5/runs` เป็นลิงก์ไปยังโฟลเดอร์ใน Drive ผลจึงไม่หายถ้ารันไทม์หลุด

In [6]:
os.makedirs(DRIVE_RUNS, exist_ok=True)
runs = "/content/repo/Round 5/runs"
!rm -rf "{runs}"
os.symlink(DRIVE_RUNS, runs)
print(os.path.islink(runs), "->", os.readlink(runs))

True -> /content/drive/MyDrive/Deep Learning/round5_runs


## 7) เทรน

ตั้งค่าเหมือนที่ใช้ใน Round 4/5: ขนาด 96x96, ไม่มี maxpool, augmentation เปิด (ค่าเริ่มต้น)
ต้องเห็นบรรทัด `ใช้ device: cuda` ตอนเริ่ม ถ้าเป็น `cpu` แปลว่าไม่ได้เลือก GPU
รอบที่สอง: เปลี่ยน `SEED` และ `RUN_NAME` แล้วรันเซลล์นี้ใหม่ (early stopping patience 10 เหมือนรอบก่อน)

In [7]:
os.environ.update({
    "IMG_SIZE": "96",
    "NO_MAXPOOL": "1",
    "SEED": "1",
    "RUN_NAME": "img96_nomp_aug_s1",
    "NUM_WORKERS": "2",        # Colab มี CPU น้อย
})
%cd "/content/repo/Round 5"
!python -u TrainingCNN.py

/content/repo/Round 5
พบ 72 คลาส
Indexing images: 100% 72/72 [00:31<00:00,  2.28it/s]
รวม 63317 samples (72 คลาส)
Train: 50653, Val: 12664
Estimating mean/std: 100% 3000/3000 [00:01<00:00, 2121.61it/s]
mean=0.5568, std=0.4390 (ประเมินจาก 3000 ภาพตัวอย่างของ train)
บันทึก normalization stats ไว้ที่ /content/repo/Round 5/runs/img96_nomp_aug_s1/norm_stats.json
RUN: img96_nomp_aug_s1 (IMG_SIZE=96, maxpool=off, optimizer=adam, augment=on)
ใช้ device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 222MB/s]
ใช้ optimizer: adam
DataLoader num_workers=2
Training:   0% 0/60 [00:00<?, ?it/s]Epoch 1/60 - train_loss: 0.7364 - train_acc: 86.38% - val_loss: 0.1375 - val_acc: 96.06% - lr: 0.000100
Training:   2% 1/60 [01:48<1:46:44, 108.55s/it]Epoch 2/60 - train_loss: 0.1469 - train_acc: 95.29% - val_loss: 0.0983 - val_acc: 96.45% - lr: 0.000100
Training:   3% 2/60 [03:48<1:51:17, 

## 8) ดูผลสรุป

In [8]:
import json
m = json.load(open(f"{runs}/{os.environ['RUN_NAME']}/metrics.json", encoding="utf-8"))
print("epochs:", m["epochs_run"], "| best val_acc: %.2f%% ที่ epoch %d" % (m["best_val_acc"] * 100, m["best_epoch"]),
      "| จบแล้ว:", m["finished"], "| %.1f นาที/epoch" % (m["sec_per_epoch"] / 60))
print("ไฟล์ผล:", os.listdir(f"{runs}/{os.environ['RUN_NAME']}"))

epochs: 55 | best val_acc: 98.53% ที่ epoch 45 | จบแล้ว: True | 2.0 นาที/epoch
ไฟล์ผล: ['norm_stats.json', 'metrics.json', 'model.pt']
